<a href="https://colab.research.google.com/github/nikhilmooloo02-stack/CLARIX_AI_AGENT/blob/main/CLARIX_AI_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# CELL 1 - Install dependencies
!pip install anthropic gradio chromadb pypdf2 pillow requests -q
print("All packages installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 837.5/837.5 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 60.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [2]:
# CELL 2 - Connect to Claude AI
import anthropic
from google.colab import userdata

# Get API key securely
client = anthropic.Anthropic(api_key=userdata.get('ANTHROPIC_API_KEY'))

# Test connection
def test_connection():
    print(client.messages.create(
      model="claude-sonnet-4-5",
      max_tokens=100,
      messages=[{"role": "user", "content": "Say: HI MY NAME IS CLARIX, YOUR AI ASSISTANT. HOW CAN I HELP YOU TODAY?."}]
    ).content[0].text)

test_connection()


HI MY NAME IS CLARIX, YOUR AI ASSISTANT. HOW CAN I HELP YOU TODAY?


In [3]:
SYSTEM_PROMPT = """You are CLARIX, a professional AI customer support agent
for South African SMEs. Your job is to:
- Handle customer complaints with empathy
- Provide clear solutions and timelines
- Escalate serious issues when necessary
- Always acknowledge the customer's frustration first
- Be concise and professional"""

conversation_history = []

def ask_clarix(user_message):
    conversation_history.append({
        "role": "user",
        "content": user_message
    })

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=conversation_history
    )

    reply = response.content[0].text
    conversation_history.append({
        "role": "assistant",
        "content": reply
    })

    print(f"\nCLARIX: {reply}\n")

# Test it
ask_clarix("A customer says their order arrived damaged and wants a refund.")


CLARIX: I understand how frustrating it must be to receive a damaged order. I'm sorry this happened, and I'll help you resolve this right away.

**Here's what I can do for you:**

1. **Immediate refund** - I can process a full refund to your original payment method within 24-48 hours

2. **Replacement option** - Or, if you'd prefer, I can send a replacement item with expedited shipping at no extra cost

**What I need from you:**
- A photo of the damaged item (if possible)
- Your order number

**Next steps:**
- Once you provide the order number, I'll initiate your preferred solution immediately
- You won't need to return the damaged item
- You'll receive a confirmation email within 1 hour

Which option would work better for you - the refund or replacement? And could you share your order number so I can get this started?

Again, I apologize for the inconvenience. We'll make this right.



In [4]:
# Cell 4 — Install Excel reader
!pip install openpyxl pandas -q
print("Excel reader ready")


Excel reader ready


In [ ]:
# Cell 5 — Load ShaNeal Product List into CLARIX
import pandas as pd
import chromadb
from google.colab import files

# Step 1 — Upload your Excel file
print("Please upload your Excel product list...")
uploaded = files.upload()

# Step 2 — Read the Excel file
filename = list(uploaded.keys())[0]
df = pd.read_excel(filename)

print(f"Found {len(df)} products")
print(f"Columns detected: {list(df.columns)}")
print(df.head(3))  # Shows first 3 rows so you can verify it loaded correctly


Please upload your Excel product list...


In [ ]:
# Cell 4 — CLARIX Gradio UI
import gradio as gr

def chat(message, history):
    conversation = []
    for human, assistant in history:
        conversation.append({"role": "user", "content": human})
        conversation.append({"role": "assistant", "content": assistant})

    conversation.append({"role": "user", "content": message})

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=500,
        system=SYSTEM_PROMPT,
        messages=conversation
    )

    return response.content[0].text

demo = gr.ChatInterface(
    fn=chat,
    title="CLARIX — AI Customer Support Agent",
    description="Professional AI-powered customer support for your business.",
    examples=[
        "My order arrived damaged and I want a refund.",
        "I've been waiting 3 weeks and nobody is responding.",
        "I want to cancel my order immediately.",
    ],
    theme=gr.themes.Soft()
)

demo.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f1143e5bbe38e84b6a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
!git clone https://github.com/nikhilmooloo02-stack/CLARIX_AI_AGENT.git

Cloning into 'CLARIX_AI_AGENT'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (6/6), done.
